In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import intrasom

# ------------------------- Einstellungen -------------------------
pd.set_option('display.max_columns', None)

In [ ]:
# ------------------------- Basisverzeichnis (aktuelles Notebook-Verzeichnis) -------------------------
base_dir = Path.cwd()
preprocessing_root = base_dir.parent.parent / "3.1_Preprocessing" / "Preprocessing"

# ------------------------- Setzen des neuesten Preprocessing-Ordners -------------------------
timestamp_folders = [f for f in preprocessing_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Preprocessing-Ordner in {preprocessing_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Verwendeter Preprocessing-Ordner: {latest_folder.name}")

# ------------------------- Neuesten Zeitstempel finden und setzen -------------------------
try:
    prep_ts = datetime.strptime(latest_folder.name, "%Y-%m-%d_%H-%M-%S")
except ValueError:
    print("Warnung: Preprocessing-Ordnername entspricht nicht dem Schema. Nutze Dateisystem-Zeit.")
    prep_ts = datetime.fromtimestamp(latest_folder.stat().st_mtime)
 
# ------------------------- Lade die preprozessierten Daten -------------------------
input_path_prep = latest_folder / "Preprocessed_SOM_Ready.csv"
df_prep = pd.read_csv(input_path_prep, low_memory=False)
df_full = df_prep


<div class="alert alert-info">
    <b>IntraSOM Machine Learning</b><br><br>
    <b>Datenquelle:</b><br>
    - Preprocessed Data: Ionen (Log + Gaussian Scaling) + Temperatur (Bereinigt)<br>
    <br>
    <b>Filter:</b><br>
    - <b>Nur Temperaturen > 10 °C</b> werden berücksichtigt.
    <br>
    <b>Bibliothek:</b><br>
    - <a href="https://github.com/InTRA-USP/IntraSOM">IntraSOM</a>
</div>

In [ ]:
# ------------------------- Features auswählen (vorerst Hauptionen) -------------------------
target_ions = ["Na", "Ca", "Mg", "Cl", "HCO3", "SO4"]
features = []
for col in df_full.columns:
    if "_gauss" in col:
        element = col.split("_")[0]
        if element in target_ions:
            features.append(col)

print(f"Input Features für SOM: {features}")

# ------------------------- Filtern nach Temperatur -------------------------
df_som = df_full[features + ['temperature_in_c']].copy()

# ------------------------- Temperatur sicherheitshalber komplett in numerisch umwandeln -------------------------
df_som['temperature_in_c'] = pd.to_numeric(df_som['temperature_in_c'], errors='coerce')

# ------------------------- Nur Temperaturen > 10°C behalten -------------------------
len_before_temp = len(df_som)
df_som = df_som[df_som['temperature_in_c'] > 10]
print(f"Filter Temp > 10°C: {len_before_temp - len(df_som)} Zeilen entfernt. Verbleibend: {len(df_som)}")

initial_len = len(df_som)
df_som.dropna(subset=features, inplace=True)
print(f"Zeilen mit fehlenden Features entfernt: {initial_len - len(df_som)}. Final: {len(df_som)}")

In [ ]:
data_values = df_som[features].values
print(f"Data Shape für IntraSOM: {data_values.shape}")

In [ ]:
# ------------------------- IntraSOM Training -------------------------
if 'som' in locals():
    del som 

# ------------------------- Map Konfiguration (X mal Y) -------------------------
mapsize = (10, 10)

# ------------------------- SOM Erstellung -------------------------
print("Erstelle IntraSOM Modell...")
som = intrasom.SOMFactory.build(
    data=data_values,
    mapsize=mapsize,
    mapshape='toroid',    # GitHub Beispiel
    lattice='hexa',       # Hexagonal 
    normalization=None,   # Features sind schon preprocessed
    initialization='random',
    neighborhood='gaussian',
    training='batch',
    name='IntraSOM_Analysis',
    component_names=features,
    missing=False         # NaNs wurden bereits entfernt
)

print("Starte Training...")
som.train()

In [ ]:
print("Training abgeschlossen. Generiere Visualisierungen...")
print("Plotte U-Matrix...")
som.plot_umatrix(show=True, print_out=True)